# 🧬 Remedia — Reseptör Odaklı İlaç Keşif Pipeline'ı (tek notebook, Colab)

Bu notebook, tüm keşif akışını **baştan sona Google Colab'da** çalıştırır:
hedef protein → bağlanma cebi → tohum moleküller → yeni molekül üretimi →
**GNINA (GPU) docking** → ADMET → sıralama → görsel sonuç. **Hiç dosya
taşımak, hiç git senkronizasyonu yapmak yok.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehmetg06/Remedia/blob/main/notebooks/remedia_pipeline.ipynb)

### 📋 Nasıl kullanılır
1. Üstten **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU (T4)** seç.
   **GNINA docking GPU olmadan çalışmaz — GPU ZORUNLUDUR.**
2. **Runtime ▸ Run all**. İlk **Hücre 0** (Miniconda kurulumu) **kernel'i yeniden
   başlatır** — bu NORMAL. Kernel yeniden başladıktan sonra **'Run all'ı TEKRAR
   çalıştır**; bu sefer Miniconda zaten kurulu olduğu için restart olmadan
   baştan sona sorunsuz akar.
3. Her hücrenin çıktısında **`✅`** işaretini gör; bir sonraki hücre bir
   öncekinin çıktısını (değişkenlerini) kullanır, sırayı bozma.

> **Parametreler:** UniProt ID, üretim yöntemi (füzyon/genetik/BRICS/random),
> molekül sayısı gibi seçimleri ilgili hücrelerin üstündeki **form kutularından**
> (dropdown/kaydırıcı) yapabilirsin. Çok satırlı SMILES listesi ise (elle tohum
> girmek istersen) kod içindeki üçlü-tırnaklı `MANUAL_SEEDS` değişkenine yapıştırılır.

> ℹ️ **Not:** Bu notebook fpocket'i conda ile kurar ve gerekli **conda Terms of
> Service'i otomatik kabul eder**. fpocket kurulamazsa docking kutusu proteinin
> geometrik merkezine kurulur (çalışır ama ideal değildir).

> 💾 **Drive kalıcılığı:** Hücre -1 Google Drive'ı en baştan bağlar. Hücre 1,
> GNINA/fpocket'i `/content/drive/MyDrive/remedia_setup/` klasöründen varsa
> **kopyalar** (indirmez/kurmaz); yoksa normal kurar ve o klasöre kopyalar.
> Sonuç: ilk çalıştırma normal sürede, **sonraki her çalıştırmada kurulum
> saniyeler içinde** biter.


## 💾 Hücre -1 — Google Drive'a bağlan (kalıcı kurulum önbelleği için)
**Ne yapıyor:** Google Drive'ı `/content/drive` altına bağlar ve
`/content/drive/MyDrive/remedia_setup/` klasörünü hazırlar. Bu klasör,
Hücre 1'in GNINA binary'sini ve fpocket'i **indirmek/kurmak yerine
kopyalayarak** kurabilmesi için kullanılır — böylece **ilk çalıştırmadan
sonraki her çalıştırmada** kurulum saniyeler içinde biter. Aynı klasör,
Hücre 8'in sonuçları kalıcı kaydetmesi için de kullanılır.
**Ne kadar sürer:** ~5–10 saniye (+ Drive erişim izni onayı, yalnızca ilk
seferde sorulur).
**Devam etmeden önce:** `✅ Google Drive bağlandı` satırını gör. Drive izni
vermezsen sorun değil — pipeline yine çalışır, yalnızca kurulum her
seferinde normal sürede yapılır ve sonuçlar oturum içinde kalır.


In [ ]:
from pathlib import Path

# Bu klasor, Hucre 1'in GNINA/fpocket kurulumunu (indirmek/kurmak yerine
# kopyalayarak) hizlica tamamlamasi ve Hucre 8'in sonuclari kalici kaydetmesi
# icin kullanilir. Kernel yeniden baslasa bile (Hucre 0) Drive mount'u VM
# seviyesinde kalir, tekrar mount etmeye gerek kalmaz (force_remount=False).
DRIVE_SETUP_DIR = Path("/content/drive/MyDrive/remedia_setup")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_SETUP_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✅ Google Drive bağlandı. Kurulum önbelleği: {DRIVE_SETUP_DIR}")
except Exception as e:
    DRIVE_SETUP_DIR = None
    print(f"ℹ️ Google Drive bağlanamadı ({e}) — Hücre 1 kurulumu her seferinde "
          "normal şekilde (indirerek) yapılacak, sonuçlar yalnızca oturum "
          "içinde kalacak.")


## 🔴 Hücre 0 — Miniconda kurulumu (⚠️ KERNEL YENİDEN BAŞLAR)
**Ne yapıyor:** fpocket'i conda ile kurabilmek için Colab'a Miniconda kurar
(Colab'da varsayılan conda yoktur). **condacolab kernel'i yeniden başlatır** —
bu beklenen, normal bir davranıştır.
**Nasıl ilerlenir:**
1. **Bu ilk hücreyi çalıştır** (ya da `Run all` — otomatik buraya kadar gelir).
2. Kernel yeniden başlayınca (*"Your session crashed / restarted"* mesajı NORMAL)
   endişelenme.
3. **`Run all`'ı TEKRAR çalıştır** — bu sefer Miniconda zaten kurulu olduğu için
   restart OLMAZ ve notebook baştan sona sorunsuz akar.
**Devam etmeden önce:** İkinci `Run all`'da bu hücrede
`✅ Miniconda zaten kurulu` yazısını gör; sonraki hücrelere geç.


In [ ]:
#@title 🔴 Hücre 0 — Miniconda kur (KERNEL YENİDEN BAŞLAR) { display-mode: "form" }
# ⚠️ BU HÜCRE KERNEL'İ YENİDEN BAŞLATIR (fpocket'i conda ile kurmak için gerekli).
# Kernel yeniden başlayınca 'Run all'ı TEKRAR çalıştır — 2. seferde restart OLMAZ,
# notebook baştan sona akar.
import sys, subprocess


def _conda_ready():
    # condacolab.check(): conda düzgün kuruluysa sessiz geçer, değilse hata verir.
    try:
        import condacolab
        condacolab.check()
        return True
    except Exception:
        return False


if _conda_ready():
    print("✅ Miniconda zaten kurulu — kernel RESTART GEREKMİYOR. Sonraki hücrelere devam et.")
else:
    print("• condacolab (Miniconda) kuruluyor... Bu işlem kernel'i YENİDEN BAŞLATACAK.")
    print("  → Restart sonrası 'Run all'ı TEKRAR çalıştır; bu sefer sorunsuz baştan sona akar.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "condacolab"], check=False)
    import condacolab
    condacolab.install()   # <-- KERNEL BURADA YENİDEN BAŞLAR


## 🔵 Hücre 1 — Kurulum
**Ne yapıyor:** GNINA (GPU'lu docking binary'si) ve **fpocket'i conda ile**
(Hücre 0'da kurulan Miniconda üzerinden — gerekli conda Terms of Service'i
otomatik kabul ederek) kurar; RDKit, meeko ve gerekli Python paketlerini kurar;
`src/` modüllerini import edebilmek için Remedia reposunu klonlayıp `src/`'yi
`sys.path`'e ekler; GPU'yu kontrol eder.

**🚀 Drive önbelleği (kalıcı kurulum):** GNINA ve fpocket için önce
`/content/drive/MyDrive/remedia_setup/` klasörüne bakılır (Hücre -1'de bağlandı).
Oradaysa **indirmek/kurmak yerine oradan kopyalanır** (çok hızlı). Yoksa normal
şekilde indirilir/kurulur, **sonra bir sonraki çalıştırma için o klasöre
kopyalanır**. Sonuç: **ilk çalıştırmada normal süre, sonraki HER çalıştırmada
kurulum saniyeler içinde biter.**
**Ne kadar sürer:** İlk çalıştırmada ~2–4 dakika (GNINA binary'si büyüktür, bir
kere iner); Drive önbelleği doluyken ~5–15 saniye.
**Devam etmeden önce:** En sonda `✅ Kurulum tamam` yazısını ve GPU satırında
`bulundu ✔` gör. `⚠️ GPU BULUNAMADI` görürsen menüden GPU seçip bu hücreyi
TEKRAR çalıştır. fpocket kurulamazsa (`⚠️ fpocket KURULAMADI`) sorun değil —
docking kutusu geometrik merkeze kurulur, pipeline yine çalışır.


In [ ]:
import os, sys, stat, subprocess, urllib.request, shutil, datetime, time
from pathlib import Path

_t0 = time.time()

# --- 1) Remedia reposunu klonla (src/ modüllerini import edebilmek için) ----
REPO_URL = "https://github.com/mehmetg06/Remedia.git"
REPO_DIR = Path("Remedia")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    print("• Remedia reposu klonlandi (Remedia/)")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=False)
    print("• Remedia reposu guncel")

SRC_DIR  = (REPO_DIR / "src").resolve()
DATA_DIR = (REPO_DIR / "data").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"• src/ import yoluna eklendi: {SRC_DIR}")

# --- 2) Python paketleri ----------------------------------------------------
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import rdkit  # noqa: F401
    print("• RDKit zaten kurulu")
except ImportError:
    print("• RDKit kuruluyor..."); pip_install("rdkit")

for mod, pkg in [("meeko", "meeko"), ("requests", "requests"),
                 ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        __import__(mod)
    except ImportError:
        print(f"• {pkg} kuruluyor..."); pip_install(pkg)

# --- 3) Drive kurulum onbellegi (Hucre -1'de mount edildiyse) ---------------
# DRIVE_SETUP_DIR Hucre -1'de tanimlanir (basariliysa Path, basarisizsa None).
# Bu hucre tek basina calistirilirsa (Hucre -1 atlandiysa) diye guvenli varsayilan:
DRIVE_SETUP_DIR = globals().get("DRIVE_SETUP_DIR", None)
if DRIVE_SETUP_DIR:
    DRIVE_SETUP_DIR = Path(DRIVE_SETUP_DIR)
    DRIVE_SETUP_DIR.mkdir(parents=True, exist_ok=True)

# --- 4) fpocket (baglanma cebi tespiti) — once Drive onbellek, yoksa conda --
FPOCKET_PATH = "/usr/local/bin/fpocket"
FPOCKET_CACHE = (DRIVE_SETUP_DIR / "fpocket_bin") if DRIVE_SETUP_DIR else None

def _fpocket_runnable(path):
    try:
        subprocess.run([path], capture_output=True, timeout=15)
        return True
    except Exception:
        return False

if shutil.which("fpocket") is None and FPOCKET_CACHE and FPOCKET_CACHE.exists():
    print(f"• fpocket Drive onbellekten KOPYALANIYOR (indirme/kurulum atlanir): {FPOCKET_CACHE}")
    shutil.copy(FPOCKET_CACHE, FPOCKET_PATH)
    os.chmod(FPOCKET_PATH, 0o755)
    if _fpocket_runnable(FPOCKET_PATH):
        print("• fpocket: onbellekten kopyalandi ✔")
    else:
        print("⚠️ Onbellekten kopyalanan fpocket calismiyor (bagimlilik eksik olabilir)"
              " — conda ile normal kuruluyor.")
        os.remove(FPOCKET_PATH)

if shutil.which("fpocket") is None:
    print("• fpocket kuruluyor (conda · bioconda)...")
    # Conda Terms of Service'i otomatik kabul et — yoksa CondaToSNonInteractiveError.
    # || true: ToS zaten kabullu/gereksizse script'i kirmasin.
    subprocess.run("conda tos accept --override-channels "
                   "--channel https://repo.anaconda.com/pkgs/main || true", shell=True)
    subprocess.run("conda tos accept --override-channels "
                   "--channel https://repo.anaconda.com/pkgs/r || true", shell=True)
    # fpocket'i bioconda'dan kur (Miniconda Hucre 0'da kuruldu).
    subprocess.run("conda install -y -c bioconda -c conda-forge fpocket "
                   ">/dev/null 2>&1 || true", shell=True)
    fpocket_found = shutil.which("fpocket")
    if fpocket_found and FPOCKET_CACHE:
        try:
            shutil.copy(fpocket_found, FPOCKET_CACHE)
            print(f"• fpocket Drive'a KOPYALANDI (bir sonraki calistirma icin): {FPOCKET_CACHE}")
        except Exception as e:
            print(f"⚠️ fpocket Drive'a kaydedilemedi: {e}")

if shutil.which("fpocket"):
    print("• fpocket: bulundu ✔", shutil.which("fpocket"))
else:
    print("⚠️ fpocket KURULAMADI — docking kutusu proteinin GEOMETRIK MERKEZINE"
          " kurulacak (Hucre 2). Bu ideal degil ama CALISIR; daha iyi pocket"
          " tespiti icin fpocket/conda kurulumunu kontrol et.")

# --- 5) GNINA GPU binary'si — once Drive onbellek, yoksa indir -------------
GNINA_PATH = "/usr/local/bin/gnina"
GNINA_CACHE = (DRIVE_SETUP_DIR / "gnina") if DRIVE_SETUP_DIR else None
GNINA_URLS = [
    "https://github.com/gnina/gnina/releases/download/v1.3/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.1/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.0.3/gnina",
]
if os.path.exists(GNINA_PATH) and os.path.getsize(GNINA_PATH) > 1_000_000:
    print("• GNINA zaten indirilmis (bu oturumda)")
elif GNINA_CACHE and GNINA_CACHE.exists() and GNINA_CACHE.stat().st_size > 1_000_000:
    print(f"• GNINA Drive onbellekten KOPYALANIYOR (indirme atlanir): {GNINA_CACHE}")
    shutil.copy(GNINA_CACHE, GNINA_PATH)
    os.chmod(GNINA_PATH, 0o755)
    print("• GNINA: onbellekten kopyalandi ✔")
else:
    subprocess.run("apt-get -qq install -y libboost-all-dev >/dev/null 2>&1", shell=True)
    ok = False
    for url in GNINA_URLS:
        try:
            print(f"• GNINA indiriliyor: {url}")
            urllib.request.urlretrieve(url, GNINA_PATH)
            if os.path.getsize(GNINA_PATH) > 1_000_000:
                ok = True; break
        except Exception as e:
            print(f"  (bu surum alinamadi: {e})")
    if not ok:
        raise RuntimeError("GNINA binary'si indirilemedi — bu hucreyi tekrar calistir.")
    m = os.stat(GNINA_PATH).st_mode
    os.chmod(GNINA_PATH, m | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    if GNINA_CACHE:
        try:
            shutil.copy(GNINA_PATH, GNINA_CACHE)
            print(f"• GNINA Drive'a KOPYALANDI (bir sonraki calistirma icin): {GNINA_CACHE}")
        except Exception as e:
            print(f"⚠️ GNINA Drive'a kaydedilemedi: {e}")

ver = subprocess.run([GNINA_PATH, "--version"], capture_output=True, text=True)
print("• GNINA:", ((ver.stdout or "") + (ver.stderr or "")).strip().splitlines()[0] if (ver.stdout or ver.stderr) else "kuruldu")

# --- 6) GPU kontrolu --------------------------------------------------------
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode == 0 and "NVIDIA-SMI" in (gpu.stdout or ""):
    print("• GPU: bulundu ✔")
else:
    print("⚠️ GPU BULUNAMADI! Runtime > Change runtime type > GPU (T4) sec, "
          "sonra BU HUCREYI TEKRAR calistir. GNINA docking (Hucre 5) GPU ZORUNLU.")

# --- 7) Ortak yollar (tum hucreler bunlari kullanir) ------------------------
RUN_ID = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
RESULTS_DIR = Path("results") / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✅ Kurulum tamam ({time.time() - _t0:.1f} sn). RUN_ID={RUN_ID}  ·  sonuc klasoru: {RESULTS_DIR}")

## 🔵 Hücre 2 — Hedef: yapı indir + bağlanma cebi bul
**Ne yapıyor:** `UNIPROT_ID` için AlphaFold yapısını **REST API'den** indirir
(`fetch_structure.py` mantığı — sabit URL değil), `fpocket` ile en yüksek
druggability'li bağlanma cebini bulup merkezini otomatik hesaplar. fpocket
yoksa/başarısızsa proteinin **geometrik merkezine** düşer (çalışır ama ideal değil).
**Parametreler:** Aşağıdaki **form kutusundan** `UNIPROT_ID`'yi ve docking kutusu
boyutunu seçebilirsin (başka bir hedef denemek için tek yapman gereken bu).
**Ne kadar sürer:** ~30–60 saniye.
**Devam etmeden önce:** `✅ HEDEF HAZIR` satırında reseptör yolunu ve docking
kutusu merkezini (Å) gör.


In [ ]:
#@title 🔵 Hücre 2 parametreleri — hedef & docking kutusu { display-mode: "form" }
UNIPROT_ID = "P00918"  #@param {type:"string"}
#@markdown Örnek hedefler: `P00918` (Karbonik Anhidraz II), `P30405` (CypD).
BOX_DIM = 20  #@param {type:"slider", min:10, max:30, step:1}
#@markdown Docking kutusu bir küptür; kenar uzunluğu (Å).
BOX_SIZE = (float(BOX_DIM), float(BOX_DIM), float(BOX_DIM))

import fetch_structure, pocket_detection

# 1) AlphaFold yapisini REST API'den indir (sabit URL DEGIL)
pdb_path = fetch_structure.fetch_alphafold(UNIPROT_ID)
RECEPTOR = str(pdb_path)
print(f"• Reseptor yapisi: {RECEPTOR}")

# 2) fpocket ile en yuksek druggability'li cebi bul, merkezini hesapla
CENTER = None
if shutil.which("fpocket"):
    try:
        pocket = pocket_detection.best_druggable_pocket(pdb_path)
        CENTER = tuple(round(c, 3) for c in pocket["center"])
        print(f"• En iyi cep: Pocket {pocket['pocket_number']}  "
              f"druggability={pocket['druggability']}  hacim={pocket['volume']} A^3")
    except Exception as e:
        print(f"⚠️ fpocket cep bulamadi ({e}) — geometrik merkeze dusuluyor.")

# 3) Fallback: fpocket yoksa/basarisizsa tum atomlarin agirlik merkezi
if CENTER is None:
    xs = ys = zs = 0.0; n = 0
    for line in Path(pdb_path).read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            xs += float(line[30:38]); ys += float(line[38:46]); zs += float(line[46:54]); n += 1
    CENTER = (round(xs / n, 3), round(ys / n, 3), round(zs / n, 3))
    print("⚠️ fpocket kurulamadi/cep bulunamadi — docking kutusu proteinin GEOMETRIK")
    print("   MERKEZINE kuruldu. Bu ideal degil ama CALISIR; daha iyi pocket tespiti")
    print("   icin fpocket kurulumunu kontrol et (Hucre 1).")

print(f"\n✅ HEDEF HAZIR")
print(f"   reseptor = {RECEPTOR}")
print(f"   docking kutusu merkezi (A) = {CENTER}   ·   boyut = {BOX_SIZE}")


## 🔵 Hücre 3 — Tohum moleküller
**Ne yapıyor:** `known_ligands.py` ile ChEMBL/PubChem'den hedefe ait bilinen
inhibitörleri otomatik getirir. Bulamazsa (veya kendi setini denemek istersen)
`MANUAL_SEEDS` üçlü-tırnaklı metnindeki SMILES'leri kullanır — form widget'ı
YOK, çok satırlı girdi sorunu yaşamazsın.
**Ne kadar sürer:** ~5–15 saniye.
**Devam etmeden önce:** `✅ N geçerli tohum molekül hazır` satırını ve listelenen
tohumları gör.

In [ ]:
from known_ligands import fetch_known_ligands
from rdkit import Chem

# Elle molekul girmek istersen buraya yapistir (cok satirli SMILES listesi —
# form kutusu DEGIL, ucul-tirnakli string; boylece cok satir sorunu yasamazsin).
# ChEMBL bos donerse otomatik olarak bunlar kullanilir.
# Her satir: SMILES [isim].  # ile baslayan satirlar yok sayilir.
MANUAL_SEEDS = """
NS(=O)(=O)c1ccccc1                benzenesulfonamide
CC(=O)Nc1nnc(s1)S(N)(=O)=O        acetazolamide
"""

def parse_seed_text(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        out.append(line.split()[0])
    return out

# 1) ChEMBL/PubChem'den otomatik getir
ligands, msg = fetch_known_ligands(UNIPROT_ID, max_results=5)
print(msg)

if ligands:
    seeds = [l["smiles"] for l in ligands]
    for l in ligands:
        print(f"  • {l['name']}: {l['smiles']}  ({l.get('activity', '')})")
else:
    seeds = parse_seed_text(MANUAL_SEEDS)
    print("• Elle girilen tohumlar (MANUAL_SEEDS) kullaniliyor:")
    for s in seeds:
        print("  •", s)

# Guvenlik agi + gecerlilik suzgeci
if not seeds:
    seeds = parse_seed_text(MANUAL_SEEDS) or ["NS(=O)(=O)c1ccccc1"]
seeds = [s for s in seeds if Chem.MolFromSmiles(s) is not None]

print(f"\n✅ {len(seeds)} gecerli tohum molekul hazir.")

## 🟣 Hücre 3.5 — REINVENT4 kurulumu (opsiyonel, sadece "pretrained" yöntemi için)
**Ne yapıyor:** [REINVENT4](https://github.com/MolecularAI/REINVENT4)'ü GitHub'dan klonlar, pip
ile kurar ve AstraZeneca'nın Zenodo'da yayınladığı **halka açık, önceden eğitilmiş**
"prior" ağırlığını (`reinvent.prior`) indirir. Bu, ilaç-benzeri moleküllerin genel
kimyasal "dilbilgisini" öğrenmiş bir sinir ağıdır — **herhangi bir reseptöre
yönlendirilmemiştir** ve **burada da yönlendirilmez**: hiçbir fine-tuning/RL/transfer
learning YAPILMAZ, yalnızca var olan prior'dan örnekleme ("sampling") yapılır.
Reseptöre uygunluk yine Hücre 5'teki GNINA docking + Hücre 6'daki ADMET filtresiyle
SONRADAN test edilir.

**🚀 Drive önbelleği (GNINA/fpocket ile aynı `remedia_setup/` klasörü):**
- **`reinvent.prior` ağırlık dosyası** — GNINA binary'siyle birebir aynı mantık:
  Drive'da varsa Zenodo'dan indirmeden oradan **kopyalanır**; yoksa indirilip
  Drive'a **kopyalanır**.
- **`pip install -e .` adımı** (~3-5GB, PyTorch/CUDA bağımlılıkları) için ham
  dosya kopyalama YAPILMAZ (CUDA native uzantıları sürücü/Python sürümüne bağlı
  olduğundan kırılgan olurdu). Bunun yerine pip'in **wheel önbelleği**
  (`PIP_CACHE_DIR`) Drive'a yönlendirilir: ilk seferden sonra wheel'ler ağdan
  değil Drive'dan gelir, kurulumun kendisi yine pip'in güvenli uyumluluk
  kontrolüyle yapılır.

**Ne kadar sürer / ne kadar yer kaplar:** İlk çalıştırmada ~5 dakika, ~3-5 GB indirme
(PyTorch + CUDA kütüphaneleri dahil — Colab GPU ortamında normaldir). Drive
önbelleği doluyken (wheel'ler + prior ağırlığı Drive'da) belirgin şekilde daha
hızlı biter — indirme adımları atlanır, yalnızca yerel kurulum/kopyalama kalır.

**Atlamak istersen:** Aşağıdaki `INSTALL_REINVENT4` kutusunu kapat — yalnızca Hücre
4'te `pretrained` yöntemini SEÇMEYECEKSEN bunu güvenle atlayabilirsin (diğer dört
yöntem buna hiç ihtiyaç duymaz). `pretrained`'i yine de seçersen, Hücre 4 kurulumu
otomatik olarak (bu hücreyi atlamış olsan bile) kendi kendine tamamlar.


In [ ]:
#@title 🟣 Hücre 3.5 parametreleri — REINVENT4 kurulumu { display-mode: "form" }
INSTALL_REINVENT4 = True  #@param {type:"boolean"}
#@markdown Kapatırsan bu hücre atlanır (yalnızca `pretrained` yöntemini Hücre 4'te
#@markdown SEÇMEYECEKSEN güvenle kapatabilirsin).

# DRIVE_SETUP_DIR Hucre -1'de tanimlanir; verilirse prior agirligi + pip wheel
# onbellegi GNINA/fpocket ile ayni Drive klasorunu kullanir (bkz. Hucre 1).
_drive_cache = globals().get("DRIVE_SETUP_DIR", None)

if INSTALL_REINVENT4:
    from generative_model import install_reinvent
    install_reinvent(log_fn=print, drive_cache_dir=_drive_cache)
else:
    print("⏭️  REINVENT4 kurulumu atlandi (INSTALL_REINVENT4=False). "
          "'pretrained' yontemini secersen Hucre 4 otomatik kurar.")


## 🔵 Hücre 4 — Molekül üret (yöntem seç: füzyon / genetik / BRICS / random / pretrained)
**Ne yapıyor:** Aşağıdaki **form kutusundan** yöntemi seçebilirsin:
- **fusion** — dört aşamalı füzyon motoru (geniş keşif → ön eleme → genetik opt.
  → rafinasyon). Varsayılan ve en kapsamlı. `molecule_generator.py`, tohum gerektirir.
- **genetic** — saf genetik algoritma (nesil sayısını kaydırıcıdan ayarla). `molecule_generator.py`, tohum gerektirir.
- **brics** — BRICS fragman rekombinasyonu. `molecule_generator.py`, tohum gerektirir.
- **random** — rastgele mutasyon. `molecule_generator.py`, tohum gerektirir.
- **pretrained** — `generative_model.py` ile **REINVENT4**'ün hazır, önceden eğitilmiş
  prior modelinden örnekleme. **Tohum molekül GEREKTİRMEZ** (Hücre 3'teki tohumlar bu
  yöntemde kullanılmaz) — sıfırdan, öğrendiği kimyasal "dilbilgisinden" üretir.
  Reseptöre özel HİÇBİR eğitim yapılmaz; reseptöre uygunluk yine sonraki hücrelerde
  (GNINA docking + ADMET) test edilir. İlk kullanımda Hücre 3.5 çalışmadıysa REINVENT4'ü
  burada otomatik kurar (birkaç dakika sürebilir).

fusion/genetic/brics/random için üretim sırasında GA fitness'i olarak docking yerine
QED (ilaç-benzerlik) vekilini kullanır (`docking_opts=None`) — **gerçek docking bir
sonraki hücrede GNINA/GPU ile ayrı yapılır**, çünkü Vina kurulu değildir.
**Ne kadar sürer:** ~30–90 saniye (fusion/genetic/brics/random, CPU'da); `pretrained`
kurulumu ilk seferde birkaç dakika sürebilir, örnekleme kendisi saniyeler sürer.
**Devam etmeden önce:** `✅ N molekül üretildi` satırını ve örnek molekülleri gör.
Bu moleküller (`molecules`) sonraki hücrelerde docklanır.


In [ ]:
#@title 🔵 Hücre 4 parametreleri — üretim yöntemi & molekül sayısı { display-mode: "form" }
method = "fusion"  #@param ["fusion", "genetic", "brics", "random", "pretrained"]
GENERATE_N = 20  #@param {type:"slider", min:5, max:100, step:5}
#@markdown `GENERATE_N`: docking'e girecek hedef molekül sayısı.
GA_GENERATIONS = 5  #@param {type:"slider", min:2, max:20, step:1}
#@markdown `GA_GENERATIONS`: yalnızca `genetic` yönteminde nesil sayısı.
#@markdown `pretrained` (REINVENT4) tohum molekül SORMAZ — yalnızca yukarıdaki
#@markdown `GENERATE_N` kullanılır.

from molecule_generator import (
    fusion_generation, genetic_algorithm, random_mutation,
    brics_recombination, write_smi,
)

# docking_opts=None -> uretim sirasinda GA fitness'i QED vekilidir (Vina gerekmez);
# GERCEK docking bir sonraki hucrede GNINA/GPU ile ayrica yapilir.
print(f"⚙️  Yontem: {method}  ·  hedef molekul sayisi: {GENERATE_N}\n")

if method == "fusion":
    final, mode = fusion_generation(seeds, docking_opts=None, log_fn=print)
    generated = [smi for smi, _ in final]
elif method == "genetic":
    final, mode = genetic_algorithm(
        seeds,
        generations=int(GA_GENERATIONS),
        population_size=max(int(GENERATE_N), 15),
        docking_opts=None,
        log_fn=print,
    )
    generated = [smi for smi, _ in final]
elif method == "brics":
    generated = brics_recombination(seeds, n=int(GENERATE_N))
elif method == "random":
    generated = random_mutation(seeds, n=int(GENERATE_N))
elif method == "pretrained":
    # Tohum molekul GEREKTIRMEZ: REINVENT4'un onceden egitilmis, receptore ozel
    # OLMAYAN prior'undan sifirdan orneklenir. Receptore uygunluk sonraki
    # hucrelerde (GNINA + ADMET) test edilir.
    from generative_model import generate_with_reinvent
    smi_out = RESULTS_DIR / "generated.smi"
    generated = generate_with_reinvent(
        num_molecules=int(GENERATE_N),
        output_path=smi_out,
        prefix="mol",
        log_fn=print,
        drive_cache_dir=globals().get("DRIVE_SETUP_DIR", None),
    )
else:
    raise ValueError(f"Bilinmeyen yontem: {method}")

# Havuz GENERATE_N'den kucukse (ozellikle fusion top-5 doner) kesif molekulleriyle
# tamamla. NOT: pretrained icin bu YAPILMAZ -- o yontem tohumsuz calisir, kucuk
# kalirsa REINVENT4'ten daha fazla ornek istemek gerekir (asagida REINVENT4 zaten
# hedeften %30 fazla orneklemistir); tohum bazli yontemlerle KARISTIRILMAZ.
if method != "pretrained" and len(generated) < GENERATE_N:
    pool = set(g for g in generated if g)
    pool.update(random_mutation(seeds, n=GENERATE_N))
    pool.update(brics_recombination(seeds, n=GENERATE_N))
    generated = [g for g in pool if g]
generated = generated[:GENERATE_N]

# (isim, SMILES) listesi — tum sonraki hucreler bunu kullanir
molecules = [(f"mol_{i:03d}", smi) for i, smi in enumerate(generated, 1)]

# Kayit icin .smi yaz (molecule_generator.write_smi formatinda) -- pretrained
# generate_with_reinvent() icinde zaten kendi smi_out'unu yazdi.
if method != "pretrained":
    smi_out = RESULTS_DIR / "generated.smi"
    write_smi(generated, smi_out, prefix="mol")

print(f"\n✅ {len(molecules)} molekul uretildi ({method})  ->  {smi_out}")
for name, smi in molecules[:10]:
    print(f"  {name}: {smi}")


## 🔵 Hücre 5 — GNINA DOCKING (GPU) — iki-aşamalı fast + accurate

**Ne yapıyor:** GNINA docking motoru iki net moda sahiptir:

- **FAST SCREENING** (`--cnn fast --cnn_scoring rescore --exhaustiveness 4 --num_modes 1`)
  — tek hafif CNN modeli, düşük exhaustiveness. Amacı **hızlı eleme**;
  skorlar kaba tahmindir, nihai karar için kullanılmaz.
- **ACCURATE MODE** (`--cnn_scoring rescore --exhaustiveness 16 --num_modes 9`,
  `--cnn` **verilmez** → GNINA'nın varsayılan çok-modelli ensemble'ı kullanılır)
  — daha yavaş ama güvenilir. Amacı **nihai karar**.

Aşağıdaki form kutusundan mod seçilir. **Varsayılan davranış otomatik
iki-aşamalı akıştır:** TÜM adaylar önce FAST modda docklandıkan sonra en iyi
`TOP_FRACTION` (varsayılan top %15) ACCURATE modda **TEKRAR** docklanır. Bu,
eskiden ayrı bir script olan `validate_top_candidates.py`'nin Vina için yaptığı
işi (en iyi adayları yüksek hassasiyetle yeniden doğrulama) GNINA akışında
**pipeline'ın doğal bir parçası** yapar —ayrı bir hücre/script çalıştırmanıza
gerek yoktur. **Nihai sıralama (Hücre 7) ACCURATE skorları kullanır** — FAST
skorlar yalnızca elemede kullanılır, docking_df'te `skor_kaynagi` sütunu
hangi skorun hangi modden geldiğini gösterir (`gnina_fast` / `gnina_accurate`).

**Manuel mod:** Hızlı bir keşif için sadece `sadece_fast`, ya da tüm adayları
nihai kararla dockla için `sadece_accurate` seçilebilir (form kutusundan).

**Motor:** `src/gnina_engine.py` (`dock_with_gnina()`, `run_two_stage_screening()`)
— artık notebook'a gömülü kod değil, test edilebilir tek bir modül.

**Ne kadar sürer:** FAST tarama molekül başına birkaç saniye; ACCURATE re-dock
molekül başına FAST'in **kabaca 4–8 katı** sürer (daha yüksek exhaustiveness +
ensemble CNN) ama yalnızca top-N aday için çalıştığından toplam süre makul kalır.
Zaman aşımı YOKTUR.
**Devam etmeden önce:** Her molekül için `✅`/`❌` satırlarını, sonra
`✅ GNINA bitti` özetini gör.


In [ ]:
#@title 🔵 Hücre 5 parametreleri — GNINA docking modu { display-mode: "form" }
DOCKING_MODE = "iki_asamali"  #@param ["iki_asamali", "sadece_fast", "sadece_accurate"]
#@markdown `iki_asamali` (VARSAYILAN — önerilir): TÜM adaylar önce FAST modda hızlıca
#@markdown elenir, en iyi `TOP_FRACTION` ACCURATE modda yeniden docklanır — nihai
#@markdown sıralama accurate skorlarını kullanır.
#@markdown `sadece_fast`: TÜM adaylar sadece FAST ile dockla (hızlı keşif için —
#@markdown skorlara nihai karar için GÜVENME).
#@markdown `sadece_accurate`: TÜM adaylar sadece ACCURATE ile dockla (yavaş ama her
#@markdown molekül için güvenilir skor).
TOP_FRACTION = 0.15  #@param {type:"slider", min:0.05, max:0.5, step:0.05}
#@markdown `TOP_FRACTION`: yalnızca `iki_asamali` modda kullanılır — FAST taramadan
#@markdown geçen en iyi yüzde kaçının ACCURATE ile yeniden docklanacağı (varsayılan
#@markdown top %15). Sabit sayı istersen aşağıdaki kod hücresinde `top_n=` geçebilirsin.

import sys
sys.path.insert(0, str(SRC_DIR))
import gnina_engine

_MODE_MAP = {"iki_asamali": "auto", "sadece_fast": "fast", "sadece_accurate": "accurate"}
GNINA_MODE = _MODE_MAP[DOCKING_MODE]
print(f"⚙️  Docking modu: {DOCKING_MODE} (gnina_engine mode='{GNINA_MODE}')"
      + (f"  ·  top_fraction={TOP_FRACTION}" if GNINA_MODE == "auto" else ""))


In [ ]:
import pandas as pd

DOCK_DIR = RESULTS_DIR / "gnina_out"
DOCK_DIR.mkdir(parents=True, exist_ok=True)

_common = dict(
    receptor=RECEPTOR, center=CENTER, size=BOX_SIZE,
    gnina_path=GNINA_PATH, out_dir=DOCK_DIR, log_fn=print,
)

if GNINA_MODE == "auto":
    rows, stage_info = gnina_engine.run_two_stage_screening(
        molecules, top_fraction=TOP_FRACTION, **_common,
    )
    print(f"\n✅ GNINA bitti (iki-aşamalı): {len(stage_info['fast'])} molekül FAST "
          f"tarandı, {len(stage_info['top_ligands'])} tanesi ACCURATE ile yeniden "
          f"docklandı. Nihai sıralama ACCURATE skorlarını kullanır.")
else:
    rows, _ = gnina_engine.run_single_mode_screening(molecules, mode=GNINA_MODE, **_common)
    ok = sum(1 for r in rows if r["affinity_kcal_mol"] is not None)
    print(f"\n✅ GNINA bitti ({GNINA_MODE}): {ok}/{len(rows)} molekül skorlandı.")

docking_df = pd.DataFrame(rows)
docking_df


## 🟣 Hücre 5.5 — (OPSİYONEL) FAST vs ACCURATE karşılaştırma / benchmark

**Ne yapıyor:** Bu hücre pipeline'ın parçası DEĞİLDİR — atlayabilirsin.
Merak edip **gerçek** rakamlarla FAST ve ACCURATE modların ne kadar farklı
sürdüğünü ve skorlarının ne kadar yakın/uzak çıktığını görmek istersen,
**aynı molekül setini** (Hücre 4'te üretilenlerin ilk birkaçı) hem FAST hem
ACCURATE ile dockler, ikisini karşılaştırır (`gnina_engine.benchmark_fast_vs_accurate`).
**Ne kadar sürer:** `BENCHMARK_N` molekül × (FAST + ACCURATE) — birkaç dakika.
**Devam etmeden önce:** Alttaki özet tabloda ortalama süre oranı ve ortalama/medyan/
maksimum skor farkını gör.


In [ ]:
#@title 🟣 (OPSİYONEL) Benchmark parametreleri { display-mode: "form" }
BENCHMARK_N = 5  #@param {type:"slider", min:2, max:20, step:1}
#@markdown `BENCHMARK_N`: karşılaştırmaya girecek molekül sayısı (Hücre 4'te üretilenlerin ilki).

bench_molecules = molecules[:int(BENCHMARK_N)]
print(f"FAST vs ACCURATE karşılaştırması — {len(bench_molecules)} molekül "
      f"(her biri İKİ kez docklanacak)\n")

bench_rows, bench_summary = gnina_engine.benchmark_fast_vs_accurate(
    bench_molecules, receptor=RECEPTOR, center=CENTER, size=BOX_SIZE,
    gnina_path=GNINA_PATH, out_dir=RESULTS_DIR / "gnina_benchmark", log_fn=print,
)

import pandas as pd
bench_df = pd.DataFrame(bench_rows)
print("\n📊 ÖZET:")
for k, v in bench_summary.items():
    print(f"  {k}: {v}")

bench_csv = RESULTS_DIR / "gnina_fast_vs_accurate_benchmark.csv"
bench_df.to_csv(bench_csv, index=False)
print(f"\n✅ Karşılaştırma kaydedildi: {bench_csv}")
bench_df


## 🔵 Hücre 6 — ADMET (Lipinski / Veber)
**Ne yapıyor:** `admet_filter.py`'nin `lipinski_veber_filter`'ı ile her molekülün
MW, LogP, HBD/HBA, dönebilir bağ ve TPSA değerlerini hesaplayıp ilaç-benzerlik
filtresini uygular. Çıktı: `admet_df` DataFrame'i (`pass` sütunu geçti/kaldı).
**Ne kadar sürer:** ~1–5 saniye (offline, RDKit).
**Devam etmeden önce:** `✅ ... filtresini geçti` özet satırını gör.

In [ ]:
import admet_filter
import pandas as pd

admet_rows = [admet_filter.lipinski_veber_filter(smi, name) for name, smi in molecules]
admet_df = pd.DataFrame(admet_rows)

passed = int(admet_df["pass"].sum())
print(f"✅ ADMET: {passed}/{len(admet_df)} molekul Lipinski/Veber filtresini gecti.")
admet_df

## 🟣 Hücre 6.5 — (OPSİYONEL) admet-ai: ML-tabanlı ADMET
**Ne yapıyor:** [`admet-ai`](https://github.com/swansonk14/admet-ai) ile Hücre
6'daki basit Lipinski/Veber kuralının **ötesinde**, eğitilmiş ML modelleriyle
tahmin edilen metrikler ekler: karaciğer toksisitesi (**DILI**), mutajenite
(**AMES** testi tahmini) ve oral biyoyararlanım (**Bioavailability**). Bunlar
Hücre 6'nın sonucunu **değiştirmez**, yanına eklenir — `admet_ai_df` olarak
saklanır ve Hücre 8'de mevcutsa otomatik gösterilir.
**Varsayılan: KAPALI.** Açarsan `torch>=2.8`, `chemprop`, `lightning` kurulur
(~birkaç dakika, birkaç GB indirme — model ağırlıkları paketin içinde geldiği
için ayrıca inmiyor, ama bağımlılıklar ağır). ⚠️ `pretrained` (REINVENT4)
üretim yöntemini de AYNI oturumda kullanacaksan dikkat: REINVENT4 kendi
`torch` sürümünü kurar, admet-ai'ın `torch>=2.8` zorunluluğu bunu üzerine
yazıp GPU/CUDA uyumsuzluğuna yol açabilir (bu repo'da tam da bu tür
TensorFlow/PyTorch CUDA çakışmaları yüzünden daha önce segfault'lar
yaşanmıştı). Sadece `fusion`/`genetic`/`brics`/`random` yöntemlerini
kullanıyorsan risksizdir. Kurulum başarısız olursa pipeline bozulmaz, sadece
basit ADMET ile devam eder.
**Ne kadar sürer:** kapalıyken ~0 saniye; açıkken ~2–6 dakika (bağımlılık
kurulumu + tahmin).
**Devam etmeden önce:** `INSTALL_ADMET_AI=False` ise atlandı mesajını gör;
`True` ise ✅ tahminler hazır satırını ve tabloyu gör.

In [ ]:
#@title 🟣 (OPSİYONEL) admet-ai kurulumu { display-mode: "form" }
INSTALL_ADMET_AI = False  #@param {type:"boolean"}
#@markdown Kapalıyken bu hücre atlanır (sadece Hücre 6'daki Lipinski/Veber
#@markdown filtresi kullanılır). Açarsan admet-ai (torch/chemprop/lightning)
#@markdown kurulur -- bkz. yukarıdaki markdown notu (kurulum süresi + REINVENT4
#@markdown ile olası torch surum catismasi).

import time

admet_ai_df = None
if INSTALL_ADMET_AI:
    import subprocess, sys
    _t0 = time.time()
    try:
        print("• admet-ai kuruluyor (birkaç dakika sürebilir)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "admet-ai"], check=True)
        from admet_ai import ADMETModel

        print("• Model yükleniyor ve tahminler hesaplanıyor...")
        _model = ADMETModel()
        _names = [name for name, _ in molecules]
        _smiles = [smi for _, smi in molecules]
        _preds = _model.predict(smiles=_smiles)
        _preds.insert(0, "ligand", _names)

        admet_ai_df = _preds[["ligand", "DILI", "AMES", "Bioavailability_Ma"]].rename(columns={
            "DILI": "hepatotoxicity_risk",       # 0-1, yuksek = karaciger toksisite riski yuksek
            "AMES": "mutagenicity_risk",          # 0-1, yuksek = mutajen olma olasiligi yuksek
            "Bioavailability_Ma": "oral_bioavailability",  # 0-1, yuksek = oral bioyararlanim olasiligi yuksek
        })
        print(f"✅ admet-ai tahminleri hazir ({time.time() - _t0:.0f} sn, {len(admet_ai_df)} molekul).")
        display(admet_ai_df)
    except Exception as e:
        print(f"⚠️ admet-ai kurulamadi/calisamadi, atlaniyor (basit ADMET ile devam): {e}")
        admet_ai_df = None
else:
    print("⏭️  admet-ai atlandi (INSTALL_ADMET_AI=False). "
          "Hucre 6'daki Lipinski/Veber filtresi tek basina kullanilacak.")

## 🔵 Hücre 7 — Sırala (rank_report.py ile birleşik sıralama)
**Ne yapıyor:** Docking skorlarını ve ADMET sonuçlarını `rank_report.py`'nin
beklediği CSV formatlarında yazar, sonra `src/rank_report.py`'yi doğrudan çağırıp
birleşik sıralamayı üretir (önce ADMET'i geçenler, sonra en negatif affinity).
Çıktı: `ranked_df` ve `results/<RUN_ID>/final_ranking.csv`.
**Ne kadar sürer:** ~2 saniye.
**Devam etmeden önce:** `✅ Birleşik sıralama hazır` satırını ve sıralı tabloyu gör.

In [ ]:
import subprocess, sys
import pandas as pd

docking_csv = RESULTS_DIR / "docking_scores.csv"
admet_csv   = RESULTS_DIR / "admet_results.csv"
ranked_csv  = RESULTS_DIR / "final_ranking.csv"

# rank_report.py'nin okuyacagi tam formatlarda yaz
docking_df.to_csv(docking_csv, index=False)   # ligand, affinity_kcal_mol
admet_df.to_csv(admet_csv, index=False)       # ligand, pass, MW, LogP, violations, ...

# src/rank_report.py'yi dogrudan cagir
subprocess.run([
    sys.executable, str(SRC_DIR / "rank_report.py"),
    "--docking", str(docking_csv),
    "--admet", str(admet_csv),
    "--output", str(ranked_csv),
], check=True)

ranked_df = pd.read_csv(ranked_csv)
print(f"\n✅ Birlesik siralama hazir  ->  {ranked_csv}")
ranked_df

## 🔵 Hücre 8 — Sonuç: tablo + 2D çizim + Google Drive'a kaydet
**Ne yapıyor:** En iyi adayları tablo ve **RDKit 2D çizimleriyle** gösterir; her
molekül için `sascorer` (RDKit Contrib/SA_Score) ile **sentezlenebilirlik skoru
(SA score, 1=kolay .. 10=çok zor)** hesaplar ve tabloya + çizim etiketlerine
ekler. Çizim etiketleri artık ADMET sonucunu (geçti/kaldı, MW, LogP, ihlaller)
**kesilmeden tam** gösterir (çok satırlı etiket). Hücre 6.5'te admet-ai çalıştırıldıysa (opsiyonel) hepatotoksisite/AMES/oral biyoyararlanım tahminleri de etikete 4. satır olarak eklenir. Tüm sonuç dosyalarını
(üretilen SMILES, docking/ADMET/sıralama CSV'leri, SA score dahil aday tablosu,
çizim PNG'si) mount edilmiş **Google Drive**'da tarihli bir klasöre kaydeder —
oturum kapansa bile sonuçlar durur.
**Ne kadar sürer:** ~10–20 saniye (Drive mount izni ister).
**Devam etmeden önce:** En iyi aday tablosunu (SA score dahil) ve molekül
çizimlerini gör; `✅ Tüm sonuçlar Google Drive'a kaydedildi` satırını gör.
Drive'a izin vermezsen sonuçlar yine oturum içindeki `results/<RUN_ID>/`
klasöründedir.

In [ ]:
import shutil
import sys, os
import pandas as pd
from rdkit import Chem, RDConfig
from rdkit.Chem import Draw

# --- Sentezlenebilirlik skoru (SA score) -- RDKit Contrib/SA_Score, model egitimi
# GEREKTIRMEZ (rdkit pip paketiyle birlikte gelir). 1=kolay sentezlenir, 10=cok zor.
try:
    _sa_score_dir = os.path.join(RDConfig.RDContribDir, "SA_Score")
    if _sa_score_dir not in sys.path:
        sys.path.append(_sa_score_dir)
    import sascorer
    _sa_score_available = True
except Exception as e:
    print(f"⚠️ SA score hesaplanamiyor (sascorer yuklenemedi): {e}")
    _sa_score_available = False


def sa_score_of(smi):
    if not _sa_score_available or not smi:
        return None
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    try:
        return round(sascorer.calculateScore(m), 2)
    except Exception:
        return None


def sa_score_yorum(score):
    if score is None:
        return "?"
    if score <= 3:
        return "kolay"
    if score < 4:
        return "kolay-orta"
    if score <= 6:
        return "orta"
    if score < 7:
        return "orta-zor"
    return "zor"


TOP_K = min(6, len(ranked_df))
top = ranked_df.head(TOP_K).copy()

# ligand ismini molecules uzerinden SMILES'e esle (2D cizim + SA score icin)
smi_by_name = {name: smi for name, smi in molecules}
top["sa_score"] = top["ligand"].map(lambda lig: sa_score_of(smi_by_name.get(lig)))
top["sa_score_yorum"] = top["sa_score"].map(sa_score_yorum)

# admet-ai (Hucre 6.5, OPSIYONEL) calistirildiysa ML-tabanli metrikleri ekle --
# calistirilmadiysa admet_ai_df tanimsiz/None'dir, sessizce atlanir.
admet_ai_df = globals().get("admet_ai_df", None)
if admet_ai_df is not None and not admet_ai_df.empty:
    top = top.merge(admet_ai_df, on="ligand", how="left")

print("🏆 EN IYI ADAYLAR:")
try:
    display(top)
except NameError:
    print(top.to_string(index=False))

# 2D cizimler
mols, legends = [], []
for _, r in top.iterrows():
    smi = smi_by_name.get(r["ligand"])
    m = Chem.MolFromSmiles(smi) if smi else None
    if m is None:
        continue
    aff = r["affinity_kcal_mol"]
    aff_s = f"{aff:.2f}" if pd.notna(aff) else "—"
    admet_ok = "PASS" if bool(r.get("admet_pass", False)) else "FAIL"
    mw = r.get("MW", "-")
    logp = r.get("LogP", "-")
    violations = r.get("violations", "-")
    sa = r.get("sa_score")
    sa_s = f"SA: {sa:.1f} ({sa_score_yorum(sa)})" if sa is not None else "SA: n/a"

    ai_line = None
    if admet_ai_df is not None:
        hep, ames, bio = r.get("hepatotoxicity_risk"), r.get("mutagenicity_risk"), r.get("oral_bioavailability")
        if pd.notna(hep) and pd.notna(ames) and pd.notna(bio):
            ai_line = f"AI: Hepatotox={hep:.2f} AMES={ames:.2f} OralF={bio:.2f}"

    mols.append(m)
    # Cok satirli etiket -- ADMET/SA bilgisi tek satirda sigmadigi icin (grid
    # hucresi genisligini asip KESILIYORDU); satirlara bolerek tam gosteriyoruz.
    legend_lines = [
        f"{r['ligand']}  {aff_s} kcal/mol",
        f"ADMET={admet_ok} MW={mw} LogP={logp} ihlal={violations}",
        sa_s,
    ]
    if ai_line:
        legend_lines.append(ai_line)
    legends.append("\n".join(legend_lines))

grid_path = RESULTS_DIR / "top_candidates.png"
grid_saved = False
if mols:
    # Colab'da MolsToGridImage bazen PIL Image yerine IPython Image donduruyor
    # (useSVG/returnPNG davranisina gore); .save() olmayabilir, bu yuzden tipi
    # kontrol edip uygun kaydetme yolunu seciyoruz.
    # subImgSize yukseltildi: 3 satirli etiket (ADMET + SA score) sigsin diye.
    _sub_h = 320 + (40 if admet_ai_df is not None else 0)
    grid = Draw.MolsToGridImage(mols, legends=legends, molsPerRow=3, subImgSize=(320, _sub_h), returnPNG=False)
    try:
        if hasattr(grid, "save"):
            grid.save(str(grid_path))
            grid_saved = True
        elif hasattr(grid, "data"):
            with open(grid_path, "wb") as f:
                f.write(grid.data)
            grid_saved = True
        else:
            print(f"⚠️ Gorsel kaydedilemedi: bilinmeyen grid tipi ({type(grid)}).")
    except Exception as e:
        print(f"⚠️ Gorsel kaydedilemedi (pipeline devam ediyor): {e}")
    try:
        display(grid)
    except NameError:
        if grid_saved:
            print(f"(cizim kaydedildi: {grid_path})")

# SA score dahil aday tablosunu ayrica CSV olarak kaydet
top_csv = RESULTS_DIR / "top_candidates.csv"
top.to_csv(top_csv, index=False)

# --- Google Drive'a kalici kaydet (oturum kapansa bile durur) ---------------
saved = [docking_csv, admet_csv, ranked_csv, top_csv, RESULTS_DIR / "generated.smi"]
if grid_saved and grid_path.exists():
    saved.append(grid_path)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/Remedia_results") / RUN_ID
    save_dir.mkdir(parents=True, exist_ok=True)
    for f in saved:
        if Path(f).exists():
            shutil.copy(f, save_dir / Path(f).name)
    print(f"\n✅ Tum sonuclar Google Drive'a kaydedildi: {save_dir}")
except Exception as e:
    print(f"\nℹ️ Google Drive'a kaydedilemedi ({e}).")
    print(f"   Sonuclar oturum icinde hazir: {RESULTS_DIR.resolve()}")

print("\n🎉 PIPELINE TAMAMLANDI.")
